# Feature Selection for Loan Default Prediction

This notebook performs:
- train-test split
- preprocessing
- imbalance handling using SMOTE-ENN
- feature importance analysis
- feature selection using Random Forest

In [1]:
import pandas as pd

df = pd.read_csv("loan_default_final_ready.csv")

# Correct target column
X = df.drop("loan_default", axis=1)
y = df["loan_default"]

print(y.value_counts())
print("Target Distribution:")
print(y.value_counts())

loan_default
1    52540
0    47460
Name: count, dtype: int64
Target Distribution:
loan_default
1    52540
0    47460
Name: count, dtype: int64


## Train-Test Split

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [3]:
# Separate categorical and numerical columns
cat_cols = X_train.select_dtypes(include='object').columns
num_cols = X_train.select_dtypes(exclude='object').columns

print(len(cat_cols), len(num_cols))

21 97


## Remove High Cardinality Features

In [4]:
high_card_cols = [
    col for col in cat_cols
    if X_train[col].nunique() > 50
]
# Remove categorical columns with very high unique values
print("Dropping:", high_card_cols)

X_train = X_train.drop(columns=high_card_cols)
X_test = X_test.drop(columns=high_card_cols)

# Recompute column lists
cat_cols = X_train.select_dtypes(include='object').columns
num_cols = X_train.select_dtypes(exclude='object').columns

Dropping: ['emp_title', 'issue_d', 'desc', 'zip_code', 'addr_state', 'earliest_cr_line', 'last_credit_pull_d', 'sec_app_earliest_cr_line']


## Build Preprocessing Pipeline

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer(
    transformers=[
        # Scale numerical features
        ('num', StandardScaler(), num_cols), 
        # Encode categorical features
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

## Apply Feature Transformations

In [6]:
X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)

print(X_train_processed.shape)

(80000, 211)


## Handle Class Imbalance using SMOTE-ENN

In [7]:
from imblearn.combine import SMOTEENN

smote_enn = SMOTEENN(random_state=42)

X_train_smoteenn, y_train_smoteenn = smote_enn.fit_resample(
    X_train_processed, y_train
)

print(y_train.value_counts())
print(y_train_smoteenn.value_counts())

loan_default
1    42032
0    37968
Name: count, dtype: int64
loan_default
0    24970
1    17313
Name: count, dtype: int64


## Train Random Forest for Feature Importance

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_fs = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_fs.fit(X_train_smoteenn, y_train_smoteenn)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

## Calculate Feature Importance Scores

In [9]:
import pandas as pd
import numpy as np

feature_importance = pd.DataFrame({
    "feature_index": np.arange(X_train_smoteenn.shape[1]),
    "importance": rf_fs.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance", ascending=False
).reset_index(drop=True)

## Select Most Important Features

In [10]:
feature_importance["cumulative_importance"] = (
    feature_importance["importance"].cumsum()
)

selected_feature_indices = feature_importance[
    feature_importance["cumulative_importance"] <= 0.95
]["feature_index"].tolist()

len(selected_feature_indices)

57

## Create Final Selected Feature Sets

In [11]:
X_train_selected = X_train_smoteenn[:, selected_feature_indices]
X_test_selected = X_test_processed[:, selected_feature_indices]

print(X_train_selected.shape)
print(X_test_selected.shape)

(42283, 57)
(20000, 57)


## Save Processed Training and Testing Data

In [12]:
import numpy as np

np.save("X_train_selected.npy", X_train_selected)
np.save("y_train_smoteenn.npy", y_train_smoteenn.values)
np.save("X_test_selected.npy", X_test_selected)
np.save("y_test.npy", y_test.values)

## Save Selected Feature Indices

In [13]:
import joblib

joblib.dump(selected_feature_indices, "selected_feature_indices.pkl")

['selected_feature_indices.pkl']